# 04 — Agent Tools as MCP Functions

**Purpose:** Register a whole toolbox of Unity Catalog SQL functions in one schema, so
Databricks' managed MCP server automatically exposes all of them at a single endpoint — no
custom MCP server process needed. Free Edition allows up to 20 tools per agent, so this goes
well beyond a single fixed "generate a report" function: the agent gets a menu of report
types it can offer the user, plus the RAG answerer and the applicant classifier.

### What This Notebook Does
1. Creates `ask_insurance_rag(question)` — calls the Mosaic AI Model Serving RAG endpoint
   from `src/03_rag_chain_and_deploy` via `ai_query()`.
2. Creates `list_report_types()` — a menu function. The agent calls this when someone asks
   for "a report" without saying which kind, so it can offer real choices instead of
   guessing.
3. Creates 7 report-generator functions, one per report type, each aggregating
   `docs.policy_documents` along a different dimension and writing the narrative with
   `ai_gen()`.
4. Creates `classify_customer(description)` — classifies a free-text applicant description
   into a cost-risk tier via `ai_classify()`.
5. Validates a handful of them, then prints the managed MCP endpoint URL.

### Source & Target
| Direction | Resource |
|-----------|----------|
| Source | `insurance_rag_agent.docs.policy_documents`, `insurance_rag_endpoint` |
| Target | `insurance_rag_agent.agent_tools.*` (10 UC functions) |

> **Prerequisites:** Run `src/03_rag_chain_and_deploy` first (the RAG endpoint must exist
> for `ask_insurance_rag` to have something to call).

---

In [0]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
CATALOG              = 'insurance_rag_agent'
AGENT_TOOLS_SCHEMA    = 'agent_tools'
DOCS_TABLE            = f'{CATALOG}.docs.policy_documents'
RAG_ENDPOINT_NAME     = 'insurance_rag_endpoint'

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{AGENT_TOOLS_SCHEMA}')

print(f'Schema: {CATALOG}.{AGENT_TOOLS_SCHEMA}')
print(f'RAG endpoint: {RAG_ENDPOINT_NAME}')

---

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.ask_insurance_rag(
  question STRING COMMENT 'A natural-language question about the insurance policy data'
)
RETURNS STRING
COMMENT 'Answers a question about the insurance policy data using the deployed Mosaic AI Model Serving RAG endpoint.'
RETURN ai_query('insurance_rag_endpoint', CAST(question AS STRING))

---

### The report menu

Rather than one `generate_report(filters...)` function the agent has to guess the shape of,
`list_report_types` gives it something to hand back to the user ("here's what I can build —
which one?"), and each report type is its own small, single-purpose tool. That split is what
makes "ask the user what they want, then build it" actually work in practice — the model
reads the menu, states it in the chat, and only calls a generator once the user has picked.

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.list_report_types()
RETURNS STRING
COMMENT 'Lists the analysis report types available, for presenting as choices to the user before generating one.'
RETURN concat(
  '1. regional_cost_report — average charges, BMI, and age across all US regions.\n',
  '2. smoker_comparison_report — smokers vs. non-smokers on cost and health metrics.\n',
  '3. age_bracket_report — charges and high-cost-policy rate by age group.\n',
  '4. family_size_report — how number of dependents relates to charges.\n',
  '5. bmi_risk_report — BMI category (Underweight/Normal/Overweight/Obese) vs. charges.\n',
  '6. high_risk_segment_report — a deep-dive profile of the highest-cost policyholders.\n',
  '7. gender_cost_report — cost patterns between male and female policyholders.'
)

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.generate_regional_cost_report()
RETURNS STRING
COMMENT 'Generates a written report comparing average charges, BMI, and age across all US regions.'
RETURN (
  WITH stats AS (
    SELECT region,
           count(*)               AS policy_count,
           round(avg(charges), 2) AS avg_charges,
           round(avg(bmi), 2)     AS avg_bmi,
           round(avg(age), 1)     AS avg_age
    FROM insurance_rag_agent.docs.policy_documents
    GROUP BY region
  ),
  summary AS (
    SELECT concat_ws(' | ', collect_list(
      concat('region=', region, ': count=', policy_count, ', avg_charges=$', avg_charges,
             ', avg_bmi=', avg_bmi, ', avg_age=', avg_age)
    )) AS stats_text
    FROM stats
  )
  SELECT ai_gen(
    CAST(concat(
      'Write a well-structured business analysis report comparing insurance policy statistics ',
      'across US regions. Call out which region has the highest and lowest average charges, ',
      'and note anything else interesting. Data: ', stats_text
    ) AS STRING)
  )
  FROM summary
)

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.generate_smoker_comparison_report()
RETURNS STRING
COMMENT 'Generates a written report comparing smokers vs. non-smokers on cost and health metrics.'
RETURN (
  WITH stats AS (
    SELECT smoker,
           count(*)               AS policy_count,
           round(avg(charges), 2) AS avg_charges,
           round(avg(bmi), 2)     AS avg_bmi,
           round(avg(age), 1)     AS avg_age
    FROM insurance_rag_agent.docs.policy_documents
    GROUP BY smoker
  ),
  summary AS (
    SELECT concat_ws(' | ', collect_list(
      concat('smoker=', smoker, ': count=', policy_count, ', avg_charges=$', avg_charges,
             ', avg_bmi=', avg_bmi, ', avg_age=', avg_age)
    )) AS stats_text
    FROM stats
  )
  SELECT ai_gen(
    CAST(concat(
      'Write a clear business analysis report comparing smokers vs. non-smokers on insurance ',
      'cost and health metrics. Call out the cost multiplier between the two groups. Data: ',
      stats_text
    ) AS STRING)
  )
  FROM summary
)

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.generate_age_bracket_report()
RETURNS STRING
COMMENT 'Generates a written report breaking down insurance charges and cost-risk tier by age bracket.'
RETURN (
  WITH stats AS (
    SELECT age_bucket,
           count(*)               AS policy_count,
           round(avg(charges), 2) AS avg_charges,
           sum(CASE WHEN charges_tier = 'High' THEN 1 ELSE 0 END) AS high_tier_count
    FROM insurance_rag_agent.docs.policy_documents
    GROUP BY age_bucket
  ),
  summary AS (
    SELECT concat_ws(' | ', collect_list(
      concat('age_bracket=', age_bucket, ': count=', policy_count, ', avg_charges=$', avg_charges,
             ', high_cost_policies=', high_tier_count)
    )) AS stats_text
    FROM stats
  )
  SELECT ai_gen(
    CAST(concat(
      'Write a business analysis report describing how insurance charges and cost-risk trend ',
      'across age brackets. Note which age group carries the most high-cost policies. Data: ',
      stats_text
    ) AS STRING)
  )
  FROM summary
)

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.generate_family_size_report()
RETURNS STRING
COMMENT 'Generates a written report on how the number of dependents relates to insurance charges.'
RETURN (
  WITH stats AS (
    SELECT children,
           count(*)               AS policy_count,
           round(avg(charges), 2) AS avg_charges
    FROM insurance_rag_agent.docs.policy_documents
    GROUP BY children
  ),
  summary AS (
    SELECT concat_ws(' | ', collect_list(
      concat('dependents=', children, ': count=', policy_count, ', avg_charges=$', avg_charges)
    )) AS stats_text
    FROM stats
  )
  SELECT ai_gen(
    CAST(concat(
      'Write a short business analysis report on how the number of dependents a policyholder ',
      'has relates to their insurance charges. Data: ', stats_text
    ) AS STRING)
  )
  FROM summary
)

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.generate_bmi_risk_report()
RETURNS STRING
COMMENT 'Generates a written report correlating BMI category with insurance charges and cost-risk tier.'
RETURN (
  WITH categorized AS (
    SELECT
      CASE WHEN bmi < 18.5 THEN 'Underweight'
           WHEN bmi < 25   THEN 'Normal'
           WHEN bmi < 30   THEN 'Overweight'
           ELSE 'Obese' END AS bmi_category,
      charges, charges_tier
    FROM insurance_rag_agent.docs.policy_documents
  ),
  stats AS (
    SELECT bmi_category,
           count(*)               AS policy_count,
           round(avg(charges), 2) AS avg_charges,
           sum(CASE WHEN charges_tier = 'High' THEN 1 ELSE 0 END) AS high_tier_count
    FROM categorized
    GROUP BY bmi_category
  ),
  summary AS (
    SELECT concat_ws(' | ', collect_list(
      concat('bmi_category=', bmi_category, ': count=', policy_count, ', avg_charges=$', avg_charges,
             ', high_cost_policies=', high_tier_count)
    )) AS stats_text
    FROM stats
  )
  SELECT ai_gen(
    CAST(concat(
      'Write a business analysis report correlating BMI category (Underweight/Normal/Overweight/',
      'Obese) with insurance charges and cost-risk tier. Data: ', stats_text
    ) AS STRING)
  )
  FROM summary
)

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.generate_high_risk_segment_report()
RETURNS STRING
COMMENT 'Generates a written deep-dive profile of the highest-cost policyholder segment.'
RETURN (
  WITH stats AS (
    SELECT
      count(*)                                                                     AS policy_count,
      round(avg(charges), 2)                                                       AS avg_charges,
      round(avg(age), 1)                                                           AS avg_age,
      round(avg(bmi), 2)                                                           AS avg_bmi,
      round(100.0 * sum(CASE WHEN smoker = 'yes' THEN 1 ELSE 0 END) / count(*), 1) AS pct_smokers
    FROM insurance_rag_agent.docs.policy_documents
    WHERE charges_tier = 'High'
  ),
  top_region AS (
    SELECT region
    FROM insurance_rag_agent.docs.policy_documents
    WHERE charges_tier = 'High'
    GROUP BY region
    ORDER BY count(*) DESC
    LIMIT 1
  )
  SELECT ai_gen(
    CAST(concat(
      'Write a business analysis report profiling the highest-cost insurance policyholder ',
      'segment (top charges tier). Data: count=', stats.policy_count,
      ', avg_charges=$', stats.avg_charges, ', avg_age=', stats.avg_age,
      ', avg_bmi=', stats.avg_bmi, ', pct_smokers=', stats.pct_smokers, '%',
      ', most_common_region=', top_region.region
    ) AS STRING)
  )
  FROM stats, top_region
)

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.generate_gender_cost_report()
RETURNS STRING
COMMENT 'Generates a written report comparing insurance cost patterns between male and female policyholders.'
RETURN (
  WITH stats AS (
    SELECT sex,
           count(*)               AS policy_count,
           round(avg(charges), 2) AS avg_charges,
           round(avg(bmi), 2)     AS avg_bmi
    FROM insurance_rag_agent.docs.policy_documents
    GROUP BY sex
  ),
  summary AS (
    SELECT concat_ws(' | ', collect_list(
      concat('sex=', sex, ': count=', policy_count, ', avg_charges=$', avg_charges, ', avg_bmi=', avg_bmi)
    )) AS stats_text
    FROM stats
  )
  SELECT ai_gen(
    CAST(concat(
      'Write a brief, neutral business analysis report comparing insurance cost patterns ',
      'between male and female policyholders. Data: ', stats_text
    ) AS STRING)
  )
  FROM summary
)

---

In [0]:
%sql
CREATE OR REPLACE FUNCTION insurance_rag_agent.agent_tools.classify_customer(
  description STRING COMMENT 'A free-text description of a prospective customer (age, smoker status, BMI, region, etc.)'
)
RETURNS STRING
COMMENT 'Classifies a prospective customer description into a Low/Medium/High insurance cost-risk tier.'
RETURN ai_classify(CAST(description AS STRING), ARRAY('Low Risk', 'Medium Risk', 'High Risk'))

---

In [0]:
%sql
SELECT
  insurance_rag_agent.agent_tools.list_report_types() AS report_menu,
  insurance_rag_agent.agent_tools.classify_customer(
    '52-year-old smoker from the southeast region, BMI 34, 2 children'
  ) AS sample_classification,
  insurance_rag_agent.agent_tools.generate_smoker_comparison_report() AS sample_report

In [0]:
workspace_url = spark.conf.get('spark.databricks.workspaceUrl')
mcp_url       = f'https://{workspace_url}/api/2.0/mcp/functions/{CATALOG}/{AGENT_TOOLS_SCHEMA}'

print(f'Managed MCP endpoint for these tools: {mcp_url}')
print('10 tools total: ask_insurance_rag, list_report_types, 7 report generators, classify_customer')
print('(Free Edition allows up to 20 tools per agent — plenty of headroom to add more later.)')